<a href="https://colab.research.google.com/github/SuryamaheshPenke/Gen_AI-Notes/blob/main/Class_3_What_is_an_LLM%3F_%2B_Transformers_and_How_They_Work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Class 3 - What is an LLM? + Transformers and How They Work**
### Module 1: Prompt Engineering & LLM Mastery
---

## **Agenda**
- What an LLM actually is (the idea, before the architecture)
- NLP history: word2vec → RNN/LSTM → the sequential bottleneck that broke everything
- Attention is All You Need: what Vaswani et al. 2017 actually figured out
- Transformer architecture: self-attention mechanics, encoder-only vs decoder-only
- KV Cache: prefill phase, decode phase, and why it matters for latency
- Inference optimisations: Flash Attention, continuous batching
- Mixture of Experts (MoE): how GPT-4, Mistral, and Gemini avoid paying full compute cost
- Reasoning models: o1/o3/Claude Extended Thinking - compute at inference time

## **PART 1 - What Is an LLM**

### Context

Before we look at architecture, we need the idea.

A language model is a probability distribution over sequences of tokens. That is it. Given a sequence of tokens so far, it assigns a probability to every possible next token. The highest-probability token (or a sampled one) gets appended, and you repeat.

An LLM - a large language model - is what happens when you scale that probability machine to billions of parameters and train it on a significant fraction of human text. At sufficient scale, something unexpected happened: the model did not just learn to predict the next word. It learned to reason, translate, summarise, write code, and follow instructions - behaviours nobody explicitly trained it to do. This is called emergent behaviour, and it was a surprise.

The word "large" is doing real work here. GPT-2 (2019, 1.5B params) could write coherent paragraphs. GPT-3 (2020, 175B params) could do few-shot in-context learning. GPT-4 (2023, estimated ~1T params with MoE) could pass the bar exam. Scale changed what the models could do qualitatively, not just quantitatively.

---

### What an LLM Outputs

**Simple**
An LLM does not "think" or "know" things the way a human does. At every step, it looks at all the text so far and asks: what word comes next? It runs that calculation billions of times while training, adjusting its internal weights each time it guesses wrong. After enough training, those weights encode an enormous amount about how language works - which means they also encode a lot about how the world works, because text describes the world.

**Technical**
An LLM is a parameterised function $f_\theta$ where $\theta$ represents hundreds of billions of floating-point weights. It is trained via next-token prediction: given tokens $t_1, t_2, ..., t_n$, the model outputs a probability distribution $P(t_{n+1} | t_1...t_n)$ over a vocabulary of ~50,000-100,000 tokens. Training minimises cross-entropy loss between the predicted distribution and the actual next token across trillions of token pairs. At inference, generation is autoregressive: sample $t_{n+1}$ from the output distribution, append it to the context, repeat. This is why generation is sequential and why the quality of the context matters so much - every generated token becomes part of the input for the next step.

---

### Why Scale Works

**Simple**
Imagine you trained on every book, website, research paper, and forum post ever written in English. You would pick up not just vocabulary but how arguments are structured, how code patterns work, how facts relate to each other. Scale is just "more of that" - more text, more parameters to store what you learned, more compute to find the patterns.

**Technical**
The scaling laws literature (Kaplan et al. 2020, Hoffmann et al. 2022 "Chinchilla") showed that loss decreases predictably as a power law with model size (N), dataset size (D), and compute (C). The Chinchilla result was specific: for a given compute budget, the optimal model is smaller and trained on more tokens than previously believed - roughly N tokens per parameter. GPT-4 class models are trained on ~13T tokens. The emergent capabilities (in-context learning, chain-of-thought, multi-step reasoning) appear above certain parameter thresholds and do not extrapolate smoothly from smaller models - they appear somewhat suddenly, which is what makes LLM progress hard to predict.

## **PART 2 - NLP History: The Road to Transformers**

### Context

Transformers did not appear from nowhere. To understand why self-attention was a breakthrough, you need to understand what it replaced and why that approach broke. The history runs: bag-of-words → word2vec → RNN/LSTM → Transformer. Each step solved a real problem with the step before it.

---

### Word2Vec (2013)

**Word2vec** (Mikolov et al. 2013) learned to represent words as points in a high-dimensional space so that similar words ended up close together - the classic demo is king - man + woman = queen. Before this, words were arbitrary symbols with no relationship to each other. Word2vec gave you static embeddings: each word has exactly one vector regardless of context. "Bank" (river) and "bank" (financial) get the same vector. This is the polysemy problem that later models solved.

---

### RNNs and LSTMs (2014-2016)

**RNNs** process a sentence one word at a time, left to right, carrying a hidden state that updates with each word. The problem: that hidden state is a fixed-size vector, and information from the beginning of a long sentence gets diluted by the time you reach the end. **LSTMs** added gating mechanisms (input, forget, output gates) to control what the network keeps and discards, which helped with the vanishing gradient problem but did not solve the core issue. Both architectures are fundamentally sequential: step $t$ depends on step $t-1$, so you cannot process positions in parallel.

---

### The Sequential Bottleneck

GPUs are fast because they do thousands of operations simultaneously. RNNs/LSTMs are forced to process one token after another, each step waiting for the previous one - you are throwing away almost all of the GPU's parallelism. For a sequence of length $T$, training requires $O(T)$ sequential operations regardless of hardware. Additionally, seq2seq architectures compress the entire source sequence into a single fixed-size context vector, which Bahdanau et al. (2015) showed degrades performance sharply on sequences longer than ~30 tokens.

**The key insight that led to Transformers:** word2vec gave fixed vectors, RNNs had a sequential bottleneck, and the fixed-size context vector was an information bottleneck. Transformers solved all three by replacing recurrence with self-attention - computing relationships between all positions simultaneously, in parallel.

## **PART 3 - Attention Is All You Need (Vaswani et al. 2017)**

### Context

In 2015, Bahdanau et al. introduced attention as an add-on to seq2seq RNNs. Instead of compressing everything into one vector, the decoder could "look back" at encoder hidden states with different weights at each decoding step. Attention worked well. Translation quality improved significantly.

But attention was still bolted onto RNNs. You still had the sequential bottleneck. The computation on each token still had to wait for the previous one.

The insight in "Attention Is All You Need" (Vaswani et al. 2017) was: what if you threw away the RNN entirely and used only attention? Attention can be computed for all pairs of positions in parallel. Remove the recurrence, and you remove the sequential bottleneck. The title is the paper's thesis.

---

### The Insight

**Simple**
Instead of reading a sentence left to right and updating a running memory, a Transformer looks at all words simultaneously and lets each word decide which other words it needs to pay attention to. The word "it" in "The cat sat on the mat because it was tired" needs to look back at "cat" to understand what "it" refers to. Attention computes exactly that lookup - for every word, for every other word, at the same time.

**Technical**
The self-attention operation computes relationships between all positions in a sequence simultaneously. For a sequence of length $n$ with $d_{model}$ dimensions, this is a matrix of $n \times n$ attention weights. The computation is fully parallelisable across positions because no step depends on the output of any other step in the same layer. The cost is $O(n^2 \cdot d)$ in memory and compute - quadratic in sequence length - but for sequences up to a few thousand tokens, this is tractable on modern hardware and the parallelism gain is decisive. The paper also introduced positional encodings to compensate for the loss of sequential position information (since attention has no built-in ordering).

---

### Self-Attention Mechanics

**Simple**
For each word, self-attention asks three questions: "What am I looking for?" (Query), "What do I offer to others?" (Key), "What is my actual content?" (Value). To compute attention for a word, you compare its Query against every other word's Key - the closer the match, the more you pull in that word's Value. The result is a weighted mix of all the Values, where the weights come from how relevant each word is to the current one.

**Technical**
Given input matrix $X \in \mathbb{R}^{n \times d_{model}}$, three learnable projection matrices $W^Q, W^K, W^V$ map $X$ to query, key, and value matrices:
$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$
Attention output is:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
The division by $\sqrt{d_k}$ prevents the dot products from growing large in magnitude as dimension increases (which would push softmax into near-zero gradient regions). Multi-head attention runs $h$ independent attention operations in parallel with different learned projections, concatenates the outputs, and projects back: $\text{MHA}(X) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$. Each head can attend to different types of relationships (syntactic, semantic, coreference, etc.).

---

### Encoder-Only vs Decoder-Only

**Simple**
The original Transformer had two parts: an encoder that reads the full input and builds a rich understanding of it, and a decoder that generates the output one token at a time while looking at the encoder's output. Modern LLMs mostly dropped the encoder and use only the decoder. Why? Because if you train a decoder-only model on enough text, it learns to understand language just fine - and decoder-only is simpler to scale and run.

Encoder-only models (like BERT) are used when you need to understand/classify existing text. Decoder-only models (GPT, Claude, Gemini, Llama) are used when you need to generate text.

**Technical**
Encoder-only models (BERT, RoBERTa) use bidirectional attention: every token can attend to every other token in both directions. They are trained with masked language modelling (MLM) and next-sentence prediction. They produce contextual embeddings but do not generate text autoregressively. They are used for classification, NER, and retrieval.

Decoder-only models (GPT family, Claude, Llama, Mistral) use causal (masked) self-attention: each token can only attend to tokens at earlier positions. This constraint is required for autoregressive generation - if token $t$ could see token $t+1$ during training, the model would trivially "cheat" by copying the answer. Causal masking is implemented by setting the upper triangle of the $QK^T$ matrix to $-\infty$ before softmax. Decoder-only architectures dominate current LLM development because they are generalists: with the right prompting, the same model does classification, translation, summarisation, and code generation.

## **PART 4 - KV Cache: Prefill vs Decode**

### Context

Autoregressive generation has a performance problem. At each generation step, the model runs a full forward pass over all previous tokens to compute attention. If you have generated 500 tokens, step 501 requires computing attention over all 500 previous tokens again - even though the Keys and Values for those 500 tokens are identical to what was computed in the previous step. This is wasteful. KV cache eliminates that redundant computation.

---

### Prefill Phase

**Simple**
When you send a prompt to an LLM, the model first processes the entire prompt at once. This is fast because it is one big parallel matrix operation. During this phase, the model computes and stores the "memory" (Keys and Values) for every token in your prompt. This stored memory is the KV cache. Time to first token (TTFT) is dominated by this phase.

**Technical**
In the prefill phase, the full prompt of length $n$ is processed as a single batch. For each attention layer, the model computes $K = XW^K$ and $V = XW^V$ for all $n$ positions simultaneously. These $K$ and $V$ tensors are cached in GPU memory (VRAM). The cost is $O(n^2)$ compute and $O(n \cdot d \cdot L)$ memory where $L$ is the number of layers. This phase is compute-bound - the GPU is fully utilised processing the large matrix multiplications. For a 128K context window, this can take several seconds.

---

### Decode Phase

**Simple**
After prefill, the model generates one token at a time. For each new token, it only needs to compute Query, Key, and Value for that one new token - and then look up the cached Keys and Values for all previous tokens to compute attention. No recomputation. The cache grows by one row per step.

**Technical**
In the decode phase, each step processes a single new token. The new token's Key and Value vectors are computed and appended to the cache. Attention for the new token is computed against all cached Keys: $\text{Attention}(q_{new}, K_{cache}, V_{cache})$. This reduces per-step compute from $O(n^2)$ to $O(n \cdot d)$ per layer. The bottleneck shifts: decode steps are memory-bandwidth-bound, not compute-bound. The GPU spends most of its time reading the KV cache from VRAM into compute units, not doing arithmetic. This is why KV cache size is a critical constraint - a 70B model with 128 attention heads and 80 layers, serving a 100K context, requires tens of GB of VRAM just for the cache.

---

### Why This Matters

**Simple**
Without KV cache, a 1000-token response would require the model to redo the full attention computation 1000 times, each time re-reading all previous tokens. With KV cache, each step is fast. The difference is roughly 100x in generation speed for long contexts.

**Technical**
KV cache trades memory for compute. The cost of serving long-context requests is dominated by the memory footprint of the cache, not the compute. At batch_size=1 and 128K context, the KV cache for a 70B model can exceed 100GB, requiring multi-GPU setups just for the cache. Quantising the KV cache (to INT8 or INT4) is a common production optimisation - it reduces VRAM usage at a small quality cost. Grouped Query Attention (GQA, used in Llama 3, Mistral) reduces the number of K/V heads relative to Q heads, shrinking KV cache size proportionally without major quality loss.

## **PART 5 - Inference Optimisations**

### Context

Running LLMs at scale requires significant engineering beyond the base Transformer. Two techniques are particularly important: Flash Attention (which makes attention computation faster and more memory-efficient) and continuous batching (which makes serving infrastructure more efficient).

---

### Flash Attention

Standard attention computes a full $n \times n$ matrix of attention scores in GPU memory before applying softmax. For a 100K context, that is 10 billion numbers - it does not fit in fast GPU memory (SRAM). Flash Attention (Dao et al. 2022) computes the same result using tiling, processing blocks of the attention matrix incrementally and maintaining running statistics, so the full matrix is never materialised. IO complexity drops from $O(n^2)$ to $O(n^2 / M)$ HBM reads/writes where $M$ is SRAM size. Essentially all production inference and training now uses Flash Attention.

**Why developers care:** faster inference, longer context windows possible without running out of GPU memory.

---

### Continuous Batching

Early serving systems processed one batch at a time - start all together, wait for all to finish. If some requests finish early, those GPU slots sit idle. Continuous batching (also called iteration-level batching, used in vLLM, TGI, TensorRT-LLM) operates at the token level: after each decode step, completed sequences are evicted from the batch and new requests are inserted immediately. The batch composition changes every token step. vLLM's PagedAttention manages KV cache in fixed-size pages (like OS virtual memory) to avoid fragmentation.

**Why developers care:** 2-10x throughput improvement over static batching in production workloads - directly reduces inference cost per request.

## **PART 6 - Mixture of Experts (MoE)**

### Context

As models scaled toward 1T+ parameters, a problem emerged: every forward pass activates all parameters for every token, regardless of whether those parameters are relevant. Mixture of Experts is the architectural solution: instead of one large feed-forward network in each Transformer layer, MoE replaces it with many smaller "expert" networks (typically 8-64) and a lightweight router that picks 2 experts per token. Result: the model has 8x the parameters but each token only activates 2/8 of them. Mistral 8x7B has ~47B total parameters but ~13B active per token. GPT-4, Gemini 1.5, Mistral, and Grok all use MoE. Challenges include load balancing (routers tend to collapse onto a few favorites without regularisation) and the fact that all parameters must still be loaded into VRAM even though only a fraction activate per step.

**Why it matters:** you get a large model's knowledge capacity at a smaller model's inference cost - this is how trillion-parameter-scale models are practical to run.

## **PART 7 - Reasoning Models**

### Context

Standard LLMs generate the answer token by token in a single forward pass (with possible chain-of-thought). The compute budget is roughly fixed by the prompt length and output length. Reasoning models flip this: they spend variable, potentially large amounts of compute at inference time thinking before producing an answer. More thinking for harder problems, less for easy ones.

OpenAI's o1 (September 2024), o3, and Anthropic's Claude with Extended Thinking are the primary examples.

---

### The Core Idea

**Simple**
A standard LLM writes the answer the same way whether the question is "What is 2+2" or "Prove Fermat's Last Theorem." A reasoning model is trained to spend time thinking - generating internal reasoning steps - before committing to an answer. For hard problems, it might think for 30 seconds and generate thousands of reasoning tokens. For easy questions, a few tokens suffice. You pay for the thinking in latency and token cost, but you get significantly better accuracy on hard problems.

**Technical**
Reasoning models are trained with reinforcement learning (specifically variants of RLHF/RLAIF/GRPO) to generate extended chain-of-thought before producing a final answer. The key properties: (1) the "thinking" tokens are typically hidden from users (they appear in a separate block, not the visible response), (2) the model learns to explore multiple reasoning paths and backtrack when a path fails (analogous to tree search but done autoregressively), (3) the RL objective rewards correct final answers regardless of intermediate reasoning path, which encourages the model to find any valid reasoning strategy. DeepSeek-R1 (January 2025) demonstrated that reasoning capability can emerge from pure RL training on verifiable problems (math, code) without supervised thinking traces. The compute scaling property: unlike pretraining (fixed compute) or sampling (fixed compute per token), reasoning models can scale effective compute at inference by allowing longer thinking chains - this is called "test-time compute scaling."

---

### Trade-offs

**Simple**
Reasoning models are slower and more expensive per query than standard models. They are not always better - on simple questions, the extra thinking is wasted. They excel on problems with a clear correct answer that can be verified: math, logic, code, multi-step analysis. They do not help much on tasks like creative writing or open-ended opinions where there is no objectively correct answer.

**Technical**
The latency profile of reasoning models is fundamentally different from standard models. A 30-second thinking window might generate 2000-5000 reasoning tokens at ~50 tokens/second, adding 40-100 seconds of latency before the first visible output token. Token costs multiply accordingly. Production deployments typically expose a "thinking budget" parameter (Claude's `budget_tokens`, o1's effort levels) that caps the maximum thinking tokens to trade off quality vs latency/cost. Benchmark performance: o3 scores above 90% on AIME (AMC 10/12 level math), tasks where GPT-4 scored ~15-30%. The gap narrows on tasks without a verifiable ground truth.

# **Code Demos**

In [ ]:
# Setup - install dependencies
!pip install numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
import time
print("Setup complete")

### **Demo 1 - Attention Cost Scaling (O(n^2) visualised)**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Attention cost grows quadratically with sequence length.
# This demo makes that concrete with real numbers.

sequence_lengths = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
d_model = 128  # simplified, real models use 4096-8192

def attention_flops(n, d):
    """FLOPs for single-head attention: QK^T is (n x d)(d x n) = n^2 * d multiply-adds."""
    return 2 * n * n * d  # 2x for multiply + add

def attention_memory_bytes(n, dtype_bytes=2):
    """Bytes to store the n x n attention matrix (float16)."""
    return n * n * dtype_bytes

flops   = [attention_flops(n, d_model) for n in sequence_lengths]
memory  = [attention_memory_bytes(n) / (1024**3) for n in sequence_lengths]  # GB

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(sequence_lengths, [f / 1e9 for f in flops], 'o-', color='steelblue', linewidth=2)
ax1.set_xlabel("Sequence Length (tokens)")
ax1.set_ylabel("GFLOPs")
ax1.set_title("Attention Compute Cost")
ax1.set_xscale('log', base=2)
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)
for n, f in zip(sequence_lengths, flops):
    if n in [128, 4096, 131072]:
        ax1.annotate(f'{f/1e9:.1f}G', (n, f/1e9), textcoords="offset points", xytext=(5, 5), fontsize=9)

ax2.plot(sequence_lengths, memory, 'o-', color='darkorange', linewidth=2)
ax2.axhline(y=80, color='red', linestyle='--', linewidth=1.5, label='80GB VRAM (A100)')
ax2.axhline(y=24, color='purple', linestyle='--', linewidth=1.5, label='24GB VRAM (RTX 3090)')
ax2.set_xlabel("Sequence Length (tokens)")
ax2.set_ylabel("Memory for Attention Matrix (GB)")
ax2.set_title("Attention Memory Footprint (float16)")
ax2.set_xscale('log', base=2)
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSequence length vs attention matrix size:")
print(f"{'Length':>10} {'Attention Matrix (GB)':>25} {'GFLOPs':>12}")
print("-" * 50)
for n, m, f in zip(sequence_lengths, memory, flops):
    print(f"{n:>10,} {m:>25.4f} {f/1e9:>12.1f}")

The left chart shows compute cost doubling for every doubling of sequence length - that is the $O(n^2)$ curve. The right chart shows why Flash Attention exists: storing the full attention matrix for a 65K context requires more VRAM than most GPUs have. Flash Attention avoids materialising that matrix.

### **Demo 2 - Self-Attention from Scratch**

In [ ]:
import numpy as np

np.random.seed(42)

# A minimal self-attention implementation.
# 4 tokens, d_model=8, single head.

n_tokens = 4
d_model = 8
d_k = 8

tokens = ["The", "cat", "sat", "mat"]

# Random input embeddings (normally learned during training)
X = np.random.randn(n_tokens, d_model)

# Random projection matrices (normally learned during training)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1

# Compute Q, K, V
Q = X @ W_Q  # (4, 8)
K = X @ W_K  # (4, 8)
V = X @ W_V  # (4, 8)

# Attention scores: every token against every other token
scores = Q @ K.T / np.sqrt(d_k)  # (4, 4)

# Softmax over the last dimension
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)  # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

attn_weights = softmax(scores)  # (4, 4)

# Weighted sum of Values
output = attn_weights @ V  # (4, 8)

print("Input shape:", X.shape)
print("Q, K, V shapes:", Q.shape, K.shape, V.shape)
print("Attention scores (unnormalised):")
print(np.round(scores, 3))
print("\nAttention weights (after softmax, each row sums to 1.0):")
print(np.round(attn_weights, 3))
print("\nRow sums (must be 1.0):", np.round(attn_weights.sum(axis=1), 6))
print("\nOutput shape:", output.shape)

# Visualise attention weights
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn_weights, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(n_tokens))
ax.set_yticks(range(n_tokens))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_xlabel("Attended to (Keys)")
ax.set_ylabel("Query token")
ax.set_title("Attention Weights: who attends to whom")
for i in range(n_tokens):
    for j in range(n_tokens):
        ax.text(j, i, f"{attn_weights[i,j]:.2f}", ha='center', va='center',
                color='white' if attn_weights[i,j] > 0.5 else 'black', fontsize=10)
plt.colorbar(im)
plt.tight_layout()
plt.show()

Each row is one token's query looking at all other tokens' keys. The values in each row sum to 1.0 - they are a probability distribution over which tokens to attend to. The output for each token is a weighted average of all Value vectors, where the weights come from this matrix. With random weights, the pattern is noisy - trained models learn weights that reflect real syntactic and semantic relationships.

### **Demo 3 - KV Cache Speedup Simulation**

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

# Simulate the compute cost of generating tokens WITH and WITHOUT KV cache.
# We measure simulated FLOPs as a proxy for time, since actual GPU timing
# requires hardware. The ratio shows why KV cache is essential.

def simulate_no_kv_cache(prompt_len, n_generate):
    """Without KV cache: at step i, recompute attention over all i tokens."""
    total_ops = 0
    for step in range(n_generate):
        context_len = prompt_len + step
        # Attention cost scales as O(n^2)
        total_ops += context_len ** 2
    return total_ops

def simulate_with_kv_cache(prompt_len, n_generate):
    """With KV cache: prefill is O(n^2), then each step is O(n) - just the new token vs all cached."""
    # Prefill: full attention over prompt
    prefill_ops = prompt_len ** 2
    # Decode: each step, new token attends to all cached KVs
    decode_ops = sum(prompt_len + step for step in range(n_generate))
    return prefill_ops + decode_ops

prompt_lengths = [128, 256, 512, 1024, 2048, 4096]
n_generate = 200

results = []
for pl in prompt_lengths:
    ops_no_cache  = simulate_no_kv_cache(pl, n_generate)
    ops_kv_cache  = simulate_with_kv_cache(pl, n_generate)
    speedup       = ops_no_cache / ops_kv_cache
    results.append((pl, ops_no_cache, ops_kv_cache, speedup))

print(f"{'Prompt':>8} {'No Cache (ops)':>18} {'KV Cache (ops)':>18} {'Speedup':>10}")
print("-" * 60)
for pl, nc, kvc, sp in results:
    print(f"{pl:>8,} {nc:>18,.0f} {kvc:>18,.0f} {sp:>10.1f}x")

fig, ax = plt.subplots(figsize=(9, 5))
pls       = [r[0] for r in results]
no_cache  = [r[1] / 1e9 for r in results]
kv_cache  = [r[2] / 1e9 for r in results]
speedups  = [r[3] for r in results]

ax.bar([p - 80 for p in pls], no_cache,  width=120, label='No KV Cache', color='tomato',   alpha=0.8)
ax.bar([p + 80 for p in pls], kv_cache,  width=120, label='With KV Cache', color='seagreen', alpha=0.8)
for pl, sp in zip(pls, speedups):
    ax.text(pl, max(no_cache) * 0.05, f'{sp:.0f}x', ha='center', fontsize=9, color='black', fontweight='bold')
ax.set_xlabel("Prompt Length (tokens)")
ax.set_ylabel("Total Simulated Ops (billions), generating 200 tokens")
ax.set_title("KV Cache: Total Compute for Generating 200 Tokens")
ax.legend()
ax.set_xticks(prompt_lengths)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

The speedup grows with prompt length because without KV cache, each new generation step re-computes attention over an ever-longer context. With KV cache, each decode step only touches the new token's query against the already-stored keys and values. At a 4096-token prompt generating 200 tokens, KV cache requires roughly 20x fewer operations.

### **Demo 4 - Token Generation Timing**

We will run this demo live in Class 14 once you have your API key set up. The simulated output below shows typical gpt-4o-mini numbers. The pedagogical point is the same: TTFT (time to first token) scales with prompt length because it is dominated by the prefill phase, while decode speed stays roughly constant because KV cache eliminates per-step recomputation.

In [ ]:
# Simulated output - typical gpt-4o-mini numbers.
# We will run the live version in Class 14 after setting up the OpenAI SDK and API key.

print("Simulated results (typical gpt-4o-mini numbers):")
print()
print("  Short prompt (~10 tokens):")
print("    TTFT: ~0.3s | Total: ~1.1s | Decode: ~60 tok/s")
print()
print("  Long prompt (~1400 tokens):")
print("    TTFT: ~1.8s | Total: ~2.7s | Decode: ~58 tok/s")
print()
print("  TTFT ratio: ~6x - prefill cost is proportional to prompt length.")
print("  Decode speed: nearly identical - KV cache makes per-step cost independent of prompt length.")

Time to first token (TTFT) scales with prompt length - that is the prefill phase. Once generation begins, the per-token speed stays roughly constant regardless of prompt length - that is the KV cache eliminating recomputation. This is why long-context APIs report latency separately as TTFT and throughput (tokens/second).

# **Key Takeaways**

1. An LLM is a probability distribution over next tokens. Everything else - reasoning, translation, code - is what emerges when you train that probability machine at sufficient scale on enough text.

2. RNNs/LSTMs broke on two things: the sequential bottleneck (no GPU parallelism across time) and the fixed-size context vector (information loss on long sequences). Transformers solved both by replacing recurrence with self-attention, which is fully parallel and maintains direct connections between all positions.

3. Self-attention computes $\text{softmax}(QK^T / \sqrt{d_k})V$ for all positions simultaneously. The $O(n^2)$ cost in sequence length is the fundamental constraint that drives Flash Attention, GQA, and long-context engineering.

4. KV cache is the core inference optimisation. Prefill (processing the prompt) is $O(n^2)$ and happens once. Decode (generating each token) is $O(n)$ per step with the cache. Without it, generation latency would be unusable for long contexts.

5. MoE (Mixture of Experts) makes large models economical. Total parameters determine knowledge capacity; active parameters per token determine compute cost. GPT-4 class models achieve trillion-parameter scale while activating only ~10-15% of parameters per forward pass.

6. Reasoning models (o1/o3/Claude Extended Thinking) introduce a third compute axis: test-time compute. Standard models have fixed inference cost. Reasoning models spend variable compute thinking before answering, which significantly improves performance on verifiable hard problems at the cost of latency and token spend.

7. The 2017 Transformer paper did not just introduce a new architecture - it enabled modern LLMs by removing the sequential bottleneck that made large-scale language model training impractical. Every frontier model today is a Transformer variant.

# **Quick Check**

1. An LLM outputs a probability distribution over the vocabulary at each step. What determines which token actually appears in the output? Name two different strategies.

2. Why did RNNs struggle with long sequences even after the LSTM's gating mechanism was introduced?

3. In self-attention, what are the Queries, Keys, and Values conceptually? If you doubled the number of attention heads without changing d_model, what would you need to adjust?

4. You have a 4096-token prompt and you want to generate 500 tokens. Without KV cache, roughly how many attention operations does the decode phase require compared to with KV cache? (Order of magnitude is fine.)

5. A Mistral 8x7B MoE model has ~47B total parameters but ~13B active per token. What are the two main downsides of MoE compared to a dense model of the same active-parameter count?

6. Flash Attention produces the same mathematical output as standard attention. What exactly does it change, and why does that matter for long-context inference?

7. A user asks you to use a reasoning model (like o1) for generating marketing copy. Is this a good use case? What types of tasks do reasoning models actually improve on over standard models?

## **Quick Check - Answers**

---

*Attempt the questions above before reading these answers.*

---

**1.** The probability distribution gives the model's estimate of each token's likelihood. Two common strategies: (a) greedy decoding - always pick the highest-probability token; (b) sampling with temperature - draw from the distribution probabilistically, where temperature above 1 flattens the distribution (more random) and temperature below 1 sharpens it (more deterministic).

**2.** Even with LSTM's gates, information must still pass sequentially through every step to reach the final hidden state. For very long sequences, gradients travelling backward through hundreds of steps still weaken before reaching early positions. LSTMs cannot "jump" attention to a distant position the way transformers can with direct position-to-position attention.

**3.** Queries are "what am I looking for?", Keys are "what do I contain?", Values are "what do I return?". The attention score is the dot product of a Query with each Key; those scores weight the Values to produce the output. If you double attention heads without changing d_model, you halve the head dimension (d_k = d_model / n_heads), reducing each head's representational capacity.

**4.** Without KV cache: the decode phase recomputes attention over the full 4096-token prompt for each of the 500 new tokens - roughly 500 full forward passes. With KV cache: keys and values are computed once and stored; each decode step only runs attention for the single new token. The decode phase is roughly 500x cheaper in attention operations with KV cache.

**5.** (a) Memory: all 47B parameters must fit in memory even though only 13B are active per token - higher GPU memory requirement than a 13B dense model of the same active capacity. (b) Routing overhead: the router must assign experts per token, adding latency and training instability when some experts rarely get selected and receive fewer gradient updates.

**6.** Flash Attention reorders the matrix multiplication and softmax to stay within the GPU's fast SRAM rather than repeatedly reading from and writing to slow HBM (main GPU memory). The math is identical, but memory I/O is reduced dramatically. For long contexts this matters because standard attention's memory access grows quadratically with sequence length; Flash Attention's is nearly linear.

**7.** No - this is not a good use case. Reasoning models spend significant compute on internal chain-of-thought before answering, increasing latency and cost. They improve on tasks requiring multi-step logic: math proofs, complex code generation, planning. Marketing copy requires creativity and style, not multi-step reasoning - a standard model with a good prompt is faster and cheaper.